# Model Explainability — SHAP Analysis
Using `shap.TreeExplainer` on the best XGBoost model to explain predictions at both **global** (dataset-level) and **local** (individual-ride) levels.

In [ ]:
import sys, os
if os.path.abspath('..') not in sys.path:
    sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
from src.utils.train_utils import temporal_split
import warnings
warnings.filterwarnings('ignore')

shap.initjs()  # Enable JS visualisations in notebook

## Load Data & Model

In [ ]:
df = pd.read_csv('../data/processed/dynamic_pricing_processed.csv')
target = 'Historical_Cost_of_Ride'

df_train, df_val, df_test = temporal_split(df)

X_train = df_train.drop(columns=[target]).select_dtypes(include='number').reset_index(drop=True)
X_test  = df_test.drop(columns=[target]).select_dtypes(include='number').reset_index(drop=True)
y_test  = df_test[target].reset_index(drop=True)

model = joblib.load('../models/xgboost_best.pkl')
print(f'Features: {X_test.shape[1]}  |  Test samples: {X_test.shape[0]}')

## Compute SHAP Values

In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer(X_test)      # Explanation object
shap_matrix = shap_values.values     # (n_samples, n_features)

mean_abs = np.abs(shap_matrix).mean(axis=0)
top5 = pd.Series(mean_abs, index=X_test.columns).sort_values(ascending=False).head(5)
print('Top-5 features by mean |SHAP|:')
display(top5.to_frame('mean_|SHAP|'))

## Global Summary — Beeswarm Plot
Each dot is a single prediction. Colour = feature value (red=high, blue=low). Position on x-axis = SHAP impact on fare.

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type='dot')

## Global Summary — Bar Chart (Mean |SHAP|)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type='bar')

## Dependence Plots — Top-5 Features

In [ ]:
for feat in top5.index:
    shap.dependence_plot(feat, shap_matrix, X_test, interaction_index='auto')

## SHAP Interaction: demand_supply_ratio × hour_sin

In [ ]:
if 'demand_supply_ratio' in X_test.columns:
    shap_interact = explainer.shap_interaction_values(X_test)
    feat_a = list(X_test.columns).index('demand_supply_ratio')
    feat_b = list(X_test.columns).index('hour_sin')

    plt.figure(figsize=(8,5))
    sc = plt.scatter(X_test['demand_supply_ratio'], shap_interact[:, feat_a, feat_b],
                     c=X_test['hour_sin'], cmap='coolwarm', alpha=0.5)
    plt.colorbar(sc, label='hour_sin')
    plt.axhline(0, color='k', linestyle='--', lw=0.8)
    plt.xlabel('demand_supply_ratio')
    plt.ylabel('SHAP interaction value')
    plt.title('SHAP Interaction: demand_supply_ratio × hour_sin')
    plt.show()

## Partial Dependence Plots

In [ ]:
for feat in ['demand_supply_ratio', 'Expected_Ride_Duration', 'hour_sin']:
    if feat in X_test.columns:
        shap.partial_dependence_plot(
            feat, model.predict, X_test,
            ice=False, model_expected_value=True, feature_expected_value=True
        )

## Waterfall Plots — Low / Median / High Fare Rides

In [ ]:
y_pred_all = model.predict(X_test)
for label, idx in [
    ('Low fare',    int(np.argmin(y_pred_all))),
    ('Median fare', int(np.argsort(y_pred_all)[len(y_pred_all)//2])),
    ('High fare',   int(np.argmax(y_pred_all)))
]:
    print(f'--- {label} (sample index {idx}) ---')
    shap.waterfall_plot(shap_values[idx])

## Static Plot Artifacts
All plots above have been saved to `visualization/model_performance/shap/` by running:
```
python src/models/explain_shap.py
```
See `reports/explainability_report.md` for the business narrative.